# Mesclar bases para tratamento

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Importando os dataframes

In [ ]:
import polars as pl
df09 = pl.read_csv('/content/drive/MyDrive/POSTECH -TECH_CHALLENGE📚DATA_ANALYTCS/FASE3/Base/PNAD/PNAD_COVID_092020.csv')
df10 = pl.read_csv('/content/drive/MyDrive/POSTECH -TECH_CHALLENGE📚DATA_ANALYTCS/FASE3/Base/PNAD/PNAD_COVID_102020.csv')
df11 = pl.read_csv('/content/drive/MyDrive/POSTECH -TECH_CHALLENGE📚DATA_ANALYTCS/FASE3/Base/PNAD/PNAD_COVID_112020.csv')

Verificando colunas diferentes entre as bases

In [ ]:
def compare_columns(df, df2):
   for col in df.columns:
      if col not in df2.columns:
         print(f"Column '{col}' not found")

In [ ]:
compare_columns(df09, df10)

In [ ]:
compare_columns(df09, df11)

In [ ]:
compare_columns(df10, df09)

In [ ]:
compare_columns(df10, df11)

In [ ]:
compare_columns(df11, df09)

Column 'A006A' not found
Column 'A006B' not found
Column 'A007A' not found


In [ ]:
compare_columns(df11, df10)

Column 'A006A' not found
Column 'A006B' not found
Column 'A007A' not found


Selecionar apenas colunas que serão usadas

In [ ]:
selected_columns = [
    "Ano", "UF", "CAPITAL", "RM_RIDE", "V1008", "V1012", "V1013", "V1016", "Estrato",
    "UPA", "V1022", "V1023", "V1030", "V1031", "V1032", "posest", "A001", "A001A",
    "A001B1", "A001B2", "A001B3", "A002", "A003", "A004", "A005", "A006",
     "A007",  "A008", "A009", "B0011", "B0012", "B0013", "B0014",
    "B0015", "B0016", "B0017", "B0018", "B0019", "B00110", "B00111", "B00112",
    "B00113", "B002", "B0031", "B0032", "B0033", "B0034", "B0035", "B0036", "B0037",
    "B0041", "B0042", "B0043", "B0044", "B0045", "B0046", "B005", "B006", "B007",
    "B008", "B009A", "B009B", "B009C", "B009D", "B009E", "B009F", "B0101", "B0102",
    "B0103", "B0104", "B0105", "B0106", "B011", "C001", "C002", "C003", "C004",
    "C005", "C0051", "C0052", "C0053", "C006", "C007", "C007A", "C007B", "C007C",
    "C007D", "C007E", "C007E1", "C007E2", "C007F", "C008", "C009", "C009A", "C010",
    "C0101", "C01011", "C01012", "C0102", "C01021", "C01022", "C0103", "C0104",
    "C011A", "C011A1", "C011A11", "C011A12", "C011A2", "C011A21", "C011A22", "C012",
    "C013", "C014", "C015", "C016", "C017A", "D0011", "D0013", "D0021", "D0023",
    "D0031", "D0033", "D0041", "D0043", "D0051", "D0053", "D0061", "D0063", "D0071",
    "D0073", "E001", "E0021", "E0022", "E0023", "E0024", "F001", "F0021", "F0022",
    "F002A1", "F002A2", "F002A3", "F002A4", "F002A5", "F0061", "F006"
]

Filtrando colunas da base de dados

In [ ]:
df09 = df09.select(
    pl.col(selected_columns).cast(pl.Int64)
)

df10 = df10.select(
    pl.col(selected_columns).cast(pl.Int64)
)

df11 = df11.select(
    pl.col(selected_columns).cast(pl.Int64)
)

Gerando uma base unificada

In [ ]:
# Adiciona a coluna de identificação em cada um
df09 = df09.with_columns(pl.lit("setembro").alias("mes_referencia"))
df10 = df10.with_columns(pl.lit("outubro").alias("mes_referencia"))
df11 = df11.with_columns(pl.lit("novembro").alias("mes_referencia"))

# Agora unifica
df_concat = pl.concat([df09, df10, df11])

Validando o formato final

In [ ]:
df_concat.shape

(1149197, 146)

Salvando nova base

In [ ]:
df_concat.write_csv("df.09.10.11.csv")

# Tratamento dos dados

In [ ]:
df = pl.read_csv("df.09.10.11.csv")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Inicializa a sessão Spark
spark = SparkSession.builder.appName("DataTreatment").getOrCreate()

# Carrega o CSV (equivalente ao pl.read_csv)
df = spark.read.csv("df.09.10.11.csv", header=True, inferSchema=True)

# Usei o Dicionario disponibilizado para interpretar as colunas

In [ ]:

# 1. Inicializar a Sessão Spark
spark = SparkSession.builder \
    .appName("Tratamento_PNAD_COVID") \
    .getOrCreate()

# 2. Carregar os dados
# header=True (usa a primeira linha como nome de coluna)
# inferSchema=True (detecta automaticamente se é int, string, etc.)
df = spark.read.csv("df.09.10.11.csv", header=True, inferSchema=True)

# 3. Garantir que as colunas originais estejam em minúsculo
df = df.toDF(*[c.lower() for c in df.columns])

# 4. Dicionário de Mapeamento
mapping = {
    "ano": "ano_de_referencia",
    "uf": "uf",
    "capital": "capital",
    "rm_ride": "regiao_metropolitana_e_ride",
    "v1008": "numero_de_selecao_do_domicilio",
    "v1012": "semana_no_mes",
    "v1013": "mes_da_pesquisa",
    "v1016": "numero_da_entrevista_no_domicilio",
    "estrato": "estrato",
    "upa": "upa",
    "v1022": "situacao_domicilio",
    "v1023": "tipo_de_area",
    "v1030": "projecao_da_populacao",
    "v1031": "peso_do_domicilio_e_das_pessoas",
    "v1032": "peso_do_domicilio_e_das_pessoas_v2",
    "posest": "dominios_de_projecao",
    "a001": "numero_de_ordem",
    "a001a": "condicao_no_domicilio",
    "a001b1": "dia_de_nascimento",
    "a001b2": "mes_de_nascimento",
    "a001b3": "ano_de_nascimento",
    "a002": "idade",
    "a003": "sexo",
    "a004": "cor",
    "a005": "escolaridade",
    "a006": "frequenta_escola",
    "a007": "atividades_escolares_em_casa",
    "a008": "dias_dedicados_a_atividades_escolares",
    "a009": "tempo_gasto_em_atividades_escolares",
    "b0011": "febre",
    "b0012": "tosse",
    "b0013": "dor_garganta",
    "b0014": "dificuldade_respirar",
    "b0015": "dor_cabeca",
    "b0016": "dor_peito",
    "b0017": "nausea",
    "b0018": "nariz_entupido",
    "b0019": "fadiga",
    "b00110": "dor_olhos",
    "b00111": "perda_olfato_paladar",
    "b00112": "dor_muscular",
    "b00113": "diarreia",
    "b002": "ida_unidade_saude",
    "b0031": "providencia_ficou_em_casa",
    "b0032": "providencia_ligou_profissional_saude",
    "b0033": "providencia_remedio_conta_propria",
    "b0034": "providencia_remedio_orientacao_medica",
    "b0035": "providencia_visita_profissional_sus",
    "b0036": "providencia_visita_profissional_particular",
    "b0037": "providencia_outra",
    "b0041": "posto_publico",
    "b0042": "pronto_socorro_publico",
    "b0043": "hospital_publico",
    "b0044": "ambulatorio_privado",
    "b0045": "pronto_socorro_privado",
    "b0046": "hospital_privado",
    "b005": "ficou_internado",
    "b006": "sedado_entubado",
    "b007": "tem_plano_de_saude",
    "b008": "fez_teste",
    "b009a": "teste_swab",
    "b009b": "resultado_swab",
    "b009c": "teste_sangue_dedo",
    "b009d": "resultado_sangue_dedo",
    "b009e": "teste_sangue_coleta",
    "b009f": "resultado_sangue_coleta",
    "b0101": "diagnostico_diabetes",
    "b0102": "diagnostico_hipertensao",
    "b0103": "diagnostico_doenca_pulmao",
    "b0104": "diagnostico_doenca_cardiaca",
    "b0105": "diagnostico_depressao",
    "b0106": "diagnostico_cancer",
    "b011": "restricao_contato",
    "c001": "trabalhou_semana_passada",
    "c002": "afastado_temporariamente",
    "c003": "motivo_afastamento",
    "c004": "remuneracao_afastamento",
    "c005": "tempo_afastado",
    "c0051": "tempo_afastado_1_a_12m",
    "c0052": "tempo_afastado_1_a_2a",
    "c0053": "tempo_afastado_maior_2a",
    "c006": "mais_um_trabalho",
    "c007": "tipo_trabalho",
    "c007a": "area_de_atuacao",
    "c007b": "carteira_assinada_estatutario",
    "c007c": "cargo_ou_funcao",
    "c007d": "atividade_principal_empresa",
    "c007e": "numero_empregados",
    "c007e1": "1_a_5_empregados",
    "c007e2": "6_a_10_empregados",
    "c007f": "contrato_suspenso",
    "c008": "horas_trabalho_normal",
    "c009": "horas_trabalho_semana_passada",
    "c009a": "desejava_trabalhar_mais_horas",
    "c010": "rend_habitual_total",
    "c0101": "rend_habitual_dinheiro",
    "c01011": "faixa_rend_habitual_dinheiro",
    "c01012": "valor_habitual_dinheiro",
    "c0102": "rend_habitual_produtos",
    "c01021": "faixa_rend_habitual_produtos",
    "c01022": "valor_habitual_produtos",
    "c0103": "apenas_beneficios",
    "c0104": "nao_remunerado",
    "c011a": "rend_efetivo_total",
    "c011a1": "rend_efetivo_dinheiro",
    "c011a11": "faixa_rend_efetivo_dinheiro",
    "c011a12": "valor_efetivo_dinheiro",
    "c011a2": "rend_efetivo_produtos",
    "c011a21": "faixa_rend_efetivo_produtos",
    "c011a22": "valor_efetivo_produtos",
    "c012": "local_trabalho_habitual",
    "c013": "trabalho_remoto_semana_passada",
    "c014": "contribui_inss",
    "c015": "providencia_conseguir_trabalho",
    "c016": "motivo_nao_procurou_trabalho",
    "c017a": "gostaria_de_ter_trabalhado",
    "d0011": "rend_aposentadoria_pensao",
    "d0013": "valor_aposentadoria",
    "d0021": "rend_pensao_alimenticia_doacao",
    "d0023": "valor_pensao_doacao",
    "d0031": "rend_bolsa_familia",
    "d0033": "valor_bolsa_familia",
    "d0041": "rend_bpc_loas",
    "d0043": "valor_bpc_loas",
    "d0051": "rend_auxilio_emergencial",
    "d0053": "valor_auxilio_emergencial",
    "d0061": "rend_seguro_desemprego",
    "d0063": "valor_seguro_desemprego",
    "d0071": "rend_outros",
    "d0073": "valor_outros",
    "e001": "solicitou_emprestimo",
    "e0021": "emprestimo_banco",
    "e0022": "emprestimo_parente_amigo",
    "e0023": "emprestimo_empregador",
    "e0024": "emprestimo_outro",
    "f001": "tipo_domicilio",
    "f0021": "valor_aluguel",
    "f0022": "faixa_aluguel",
    "f002a1": "item_sabao",
    "f002a2": "item_alcool_70",
    "f002a3": "item_mascaras",
    "f002a4": "item_luvas",
    "f002a5": "item_agua_sanitaria",
    "f0061": "quem_respondeu",
    "f006": "ordem_respondente"
}

# 5. Aplicar o renomeio (Select com Alias)
# mapping.get(c, c) garante que se a coluna não estiver no dicionário, ela mantém o nome atual
df_final = df.select([F.col(c).alias(mapping.get(c, c)) for c in df.columns])

# 6. Visualizar resultado
df_final.show(5)
df_final.printSchema()

+-----------------+---+-------+---------------------------+------------------------------+-------------+---------------+---------------------------------+-------+---------+------------------+------------+---------------------+-------------------------------+----------------------------------+--------------------+---------------+---------------------+-----------------+-----------------+-----------------+-----+----+---+------------+----------------+----------------------------+-------------------------------------+-----------------------------------+-----+-----+------------+--------------------+----------+---------+------+--------------+------+---------+--------------------+------------+--------+-----------------+-------------------------+------------------------------------+---------------------------------+-------------------------------------+-----------------------------------+------------------------------------------+-----------------+-------------+----------------------+----------

# Criação de variaveis

Ida ao hospital publico ou privado

In [ ]:


# 1. Primeiro renomeie as colunas (usando o mapping que definimos antes)
df = df.toDF(*[c.lower() for c in df.columns])
df = df.select([F.col(c).alias(mapping.get(c, c)) for c in df.columns])

# 2. Agora crie a coluna condicional (com os nomes já traduzidos)
df = df.withColumn(
    "atendimento_publico",
    F.when(
        (F.col("posto_publico") == 1) |
        (F.col("pronto_socorro_publico") == 1) |
        (F.col("hospital_publico") == 1),
        1
    ).when(
        (F.col("ambulatorio_privado") == 1) |
        (F.col("pronto_socorro_privado") == 1) |
        (F.col("hospital_privado") == 1),
        0
    ).otherwise(9)
)

df.select("atendimento_publico").show(5)

+-------------------+
|atendimento_publico|
+-------------------+
|                  9|
|                  9|
|                  9|
|                  9|
|                  9|
+-------------------+
only showing top 5 rows


In [ ]:
df.where(df.atendimento_publico == 1).show(5)

+-----------------+---+-------+---------------------------+------------------------------+-------------+---------------+---------------------------------+-------+---------+------------------+------------+---------------------+-------------------------------+----------------------------------+--------------------+---------------+---------------------+-----------------+-----------------+-----------------+-----+----+---+------------+----------------+----------------------------+-------------------------------------+-----------------------------------+-----+-----+------------+--------------------+----------+---------+------+--------------+------+---------+--------------------+------------+--------+-----------------+-------------------------+------------------------------------+---------------------------------+-------------------------------------+-----------------------------------+------------------------------------------+-----------------+-------------+----------------------+----------

Resultado do teste

In [ ]:
df = df.withColumn(
    "resultado_positivo",
    F.when(
        (F.col("resultado_swab") == 1) |
        (F.col("resultado_sangue_dedo") == 1) |
        (F.col("resultado_sangue_coleta") == 1),
        1
    ).when(
        (F.col("resultado_swab") == 2) |
        (F.col("resultado_sangue_dedo") == 2) |
        (F.col("resultado_sangue_coleta") == 2),
        0
    ).otherwise(9)
)


df.select("resultado_positivo").show(5)

+------------------+
|resultado_positivo|
+------------------+
|                 9|
|                 9|
|                 9|
|                 9|
|                 9|
+------------------+
only showing top 5 rows


In [ ]:
df.where(df.resultado_positivo == 1).count()

32922

Traduzindo unidade federativa

In [ ]:
map_uf = {
    11: 'Rondônia', 12: 'Acre', 13: 'Amazonas', 14: 'Roraima', 15: 'Pará', 16: 'Amapá',
    17: 'Tocantins', 21: 'Maranhão', 22: 'Piauí', 23: 'Ceará', 24: 'Rio Grande do Norte',
    25: 'Paraíba', 26: 'Pernambuco', 27: 'Alagoas', 28: 'Sergipe', 29: 'Bahia',
    31: 'Minas Gerais', 32: 'Espírito Santo', 33: 'Rio de Janeiro', 35: 'São Paulo',
    41: 'Paraná', 42: 'Santa Catarina', 43: 'Rio Grande do Sul', 50: 'Mato Grosso do Sul',
    51: 'Mato Grosso', 52: 'Goiás', 53: 'Distrito Federal'
}

uf_map_df = spark.createDataFrame([(k, v) for k, v in map_uf.items()], ["uf", "uf_nome"])
df = df.join(uf_map_df, on="uf", how="left")

# 7. Resultado Final
df.show(5)

+---+-----------------+-------+---------------------------+------------------------------+-------------+---------------+---------------------------------+-------+---------+------------------+------------+---------------------+-------------------------------+----------------------------------+--------------------+---------------+---------------------+-----------------+-----------------+-----------------+-----+----+---+------------+----------------+----------------------------+-------------------------------------+-----------------------------------+-----+-----+------------+--------------------+----------+---------+------+--------------+------+---------+--------------------+------------+--------+-----------------+-------------------------+------------------------------------+---------------------------------+-------------------------------------+-----------------------------------+------------------------------------------+-----------------+-------------+----------------------+----------

Variaveis por região

In [ ]:
# Definindo as listas de estados
sudeste = ['São Paulo', 'Rio de Janeiro', 'Minas Gerais', 'Espírito Santo']
sul = ['Paraná', 'Santa Catarina', 'Rio Grande do Sul']
centro_oeste = ['Mato Grosso do Sul', 'Mato Grosso', 'Goiás', 'Distrito Federal']
norte = ['Rondônia', 'Acre', 'Amazonas', 'Roraima', 'Pará', 'Amapá', 'Tocantins']
nordeste = ['Maranhão', 'Piauí', 'Ceará', 'Rio Grande do Norte', 'Paraíba', 'Pernambuco', 'Alagoas', 'Sergipe', 'Bahia']

# Aplicando a lógica com F.when (Equivalente ao map_elements do Polars)
df = df.withColumn(
    'regiao',
    F.when(F.col('uf_nome').isin(sudeste), 'Sudeste')
     .when(F.col('uf_nome').isin(sul), 'Sul')
     .when(F.col('uf_nome').isin(centro_oeste), 'Centro-Oeste')
     .when(F.col('uf_nome').isin(norte), 'Norte')
     .when(F.col('uf_nome').isin(nordeste), 'Nordeste')
     .otherwise(F.col('uf_nome'))
)


In [ ]:
df.where("regiao = 'Sudeste'").show(5)

+---+-----------------+-------+---------------------------+------------------------------+-------------+---------------+---------------------------------+-------+---------+------------------+------------+---------------------+-------------------------------+----------------------------------+--------------------+---------------+---------------------+-----------------+-----------------+-----------------+-----+----+---+------------+----------------+----------------------------+-------------------------------------+-----------------------------------+-----+-----+------------+--------------------+----------+---------+------+--------------+------+---------+--------------------+------------+--------+-----------------+-------------------------+------------------------------------+---------------------------------+-------------------------------------+-----------------------------------+------------------------------------------+-----------------+-------------+----------------------+----------

Versão final tratada

In [ ]:
from pyspark.sql import SparkSession
import os
import shutil

spark = SparkSession.builder.getOrCreate()

output_path = "df_09_10_11v_tmp"
final_csv = "df_09_10_11v.csv"

(
    df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(output_path)
)

# localizar e renomear o CSV gerado
for file in os.listdir(output_path):
    if file.endswith(".csv"):
        shutil.move(
            os.path.join(output_path, file),
            final_csv
        )

# remover a pasta temporária
shutil.rmtree(output_path)


In [ ]:

output_path = "df_09_10_11v_parquet_tmp"
final_parquet = "df_09_10_11v.parquet"

# grava forçando uma única partição
(
    df
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(output_path)
)

# localizar o arquivo parquet gerado
for file in os.listdir(output_path):
    if file.endswith(".parquet"):
        shutil.move(
            os.path.join(output_path, file),
            final_parquet
        )

# remover a pasta temporária
shutil.rmtree(output_path)
